# 04履约时效与客户评分分析

本阶段基于order_base分析已送达订单的履约时效与客户评分表现，重点关注配送天数、延迟天数、延迟率、评分和1星差评率之间的关系。该阶段用于判断物流延迟是否可能影响客户满意度，并识别需要优先关注的履约风险订单。

## 1 读取数据

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data_clean")

order_base = pd.read_csv(DATA_DIR / "order_base.csv")

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_cols:
    order_base[col] = pd.to_datetime(order_base[col], errors="coerce")

print("order_base行数:", order_base.shape[0])
print("order_base列数:", order_base.shape[1])

display(order_base.head())

order_base行数: 99441
order_base列数: 28


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delay_days,...,payment_total,payment_count,payment_installments_max,payment_type_count,main_payment_type,item_count,product_count,seller_count,price_total,freight_total
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.44,-7.11,...,38.71,3.0,1.0,2.0,voucher,1.0,1.0,1.0,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.78,-5.36,...,141.46,1.0,1.0,1.0,boleto,1.0,1.0,1.0,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.39,-17.25,...,179.12,1.0,3.0,1.0,credit_card,1.0,1.0,1.0,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.21,-12.98,...,72.20,1.0,1.0,1.0,credit_card,1.0,1.0,1.0,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.87,-9.24,...,28.62,1.0,1.0,1.0,credit_card,1.0,1.0,1.0,19.90,8.72


## 2 确定分析样本
本阶段分析样本限定为已送达、配送时间字段完整且存在评分的订单。这样可以保证delivery_days、delay_days和review_score均可用于后续履约与满意度分析。

### 2.1 检查履约时间字段完整性

In [3]:
delivered_orders = order_base[
    order_base["order_status"] == "delivered"
].copy()

delivery_time_missing_check = (
    delivered_orders[
        [
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delivery_days",
            "delay_days",
            "review_score",
        ]
    ]
    .isna()
    .sum()
    .to_frame("missing_count")
)

delivery_time_missing_check["missing_rate"] = (
    delivery_time_missing_check["missing_count"] / len(delivered_orders)
).round(4)

display(delivery_time_missing_check)

,missing_count,missing_rate
order_purchase_timestamp,0,0.0000
order_delivered_customer_date,8,0.0001
order_estimated_delivery_date,0,0.0000
delivery_days,8,0.0001
delay_days,8,0.0001
review_score,646,0.0067


### 2.2 筛选履约字段和评分字段完整的订单

In [4]:
delivery_base = order_base[
    (order_base["order_status"] == "delivered")
    & order_base["delivery_days"].notna()
    & order_base["delay_days"].notna()
    & order_base["review_score"].notna()
].copy()

delivery_base["is_late"] = delivery_base["delay_days"] > 0
delivery_base["is_one_star"] = delivery_base["review_score"] == 1
delivery_base["is_low_score"] = delivery_base["review_score"] <= 2

print("已送达订单数:", order_base[order_base["order_status"] == "delivered"]["order_id"].nunique())
print("履约评分分析样本数:", delivery_base["order_id"].nunique())
print("样本order_id是否唯一:", delivery_base["order_id"].is_unique)

已送达订单数: 96478
履约评分分析样本数: 95824
样本order_id是否唯一: True


**观察与总结**

本节将分析样本限定为已送达、履约时间字段完整且存在客户评分的订单。这样可以确保每笔订单都能计算delivery_days、delay_days，并能与review_score进行关联分析。

从样本数量看，已送达订单中绝大多数可以进入履约评分分析，说明该数据集能够支撑后续关于配送时效与客户满意度的分析。少量订单由于缺少评分或关键时间字段被排除，后续结论仅针对具备完整履约和评分信息的已送达订单。

## 3 整体履约表现

In [5]:
delivery_summary = pd.Series(
    {
        "orders": delivery_base["order_id"].nunique(),
        "avg_delivery_days": delivery_base["delivery_days"].mean(),
        "median_delivery_days": delivery_base["delivery_days"].median(),
        "avg_delay_days": delivery_base["delay_days"].mean(),
        "median_delay_days": delivery_base["delay_days"].median(),
        "late_orders": delivery_base["is_late"].sum(),
        "late_rate": delivery_base["is_late"].mean(),
        "avg_review_score": delivery_base["review_score"].mean(),
        "one_star_rate": delivery_base["is_one_star"].mean(),
        "low_score_rate": delivery_base["is_low_score"].mean(),
    }
).to_frame("value")

count_metrics = ["orders", "late_orders"]
rate_metrics = ["late_rate", "one_star_rate", "low_score_rate"]

value_metrics = delivery_summary.index.difference(
    count_metrics + rate_metrics
)

delivery_summary.loc[value_metrics, "value"] = (
    delivery_summary.loc[value_metrics, "value"].round(2)
)

delivery_summary.loc[rate_metrics, "value"] = (
    delivery_summary.loc[rate_metrics, "value"].round(4)
)

display(delivery_summary)

,value
orders,95824.0000
avg_delivery_days,12.5200
median_delivery_days,10.2100
avg_delay_days,-11.2200
median_delay_days,-11.9700
late_orders,7655.0000
late_rate,0.0799
avg_review_score,4.1600
one_star_rate,0.0972
low_score_rate,0.1277


**观察与总结**

从整体履约表现看，履约评分分析样本共包含95824个订单，平均配送时长约12.52天，中位数约10.21天，说明大部分订单在10到13天左右完成送达。

delay_days的平均值和中位数均为负数，说明多数订单实际送达时间早于平台预计送达时间。整体延迟率约7.99%，表明延迟订单不是主体，但仍然存在一定规模。

从评分表现看，样本整体平均评分约4.16，1星差评率约9.72%，1到2星低分率约12.77%。这说明整体满意度较高，但仍有一部分订单产生明显负面体验。后续需要进一步比较延迟订单和准时订单之间的评分差异。

## 4 准时订单与延迟订单评分对比

In [6]:
delivery_base["delivery_status"] = np.where(
    delivery_base["is_late"],
    "late",
    "on_time_or_early",
)

late_review_comparison = (
    delivery_base
    .groupby("delivery_status", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        avg_delivery_days=("delivery_days", "mean"),
        avg_delay_days=("delay_days", "mean"),
        avg_review_score=("review_score", "mean"),
        one_star_rate=("is_one_star", "mean"),
        low_score_rate=("is_low_score", "mean"),
    )
)

late_review_comparison[
    [
        "avg_delivery_days",
        "avg_delay_days",
        "avg_review_score",
    ]
] = late_review_comparison[
    [
        "avg_delivery_days",
        "avg_delay_days",
        "avg_review_score",
    ]
].round(2)

late_review_comparison[["one_star_rate", "low_score_rate"]] = (
    late_review_comparison[["one_star_rate", "low_score_rate"]]
    .round(4)
)

display(late_review_comparison)

,delivery_status,orders,avg_delivery_days,avg_delay_days,avg_review_score,one_star_rate,low_score_rate
0,late,7655,31.39,9.45,2.57,0.4615,0.5403
1,on_time_or_early,88169,10.88,-13.01,4.29,0.0655,0.0919


**观察与总结**

本节按是否延迟将订单分为准时或提前订单和延迟订单，并比较两组的评分表现。

从结果看，准时或提前订单平均评分约4.29，而延迟订单平均评分约2.57，二者相差约1.73分；准时或提前订单的1星差评率约6.56%，而延迟订单约46.15%。这说明延迟订单不仅评分更低，而且极端差评风险显著更高。

## 5 延迟天数分组分析
观察是否延迟越久评分越差

In [7]:
delay_bins = [-np.inf, 0, 1, 3, 7, 14, np.inf]

delay_labels = [
    "on_time_or_early",
    "late_0_1d",
    "late_1_3d",
    "late_3_7d",
    "late_7_14d",
    "late_14d_plus",
]

delivery_base["delay_group"] = pd.cut(
    delivery_base["delay_days"],
    bins=delay_bins,
    labels=delay_labels,
)

delay_group_analysis = (
    delivery_base
    .groupby("delay_group", observed=True, as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        avg_delay_days=("delay_days", "mean"),
        avg_review_score=("review_score", "mean"),
        one_star_rate=("is_one_star", "mean"),
        low_score_rate=("is_low_score", "mean"),
    )
)

delay_group_analysis[
    ["avg_delay_days", "avg_review_score"]
] = delay_group_analysis[
    ["avg_delay_days", "avg_review_score"]
].round(2)

delay_group_analysis[["one_star_rate", "low_score_rate"]] = (
    delay_group_analysis[["one_star_rate", "low_score_rate"]]
    .round(4)
)

display(delay_group_analysis)

,delay_group,orders,avg_delay_days,avg_review_score,one_star_rate,low_score_rate
0,on_time_or_early,88169,-13.01,4.29,0.0655,0.0919
1,late_0_1d,1277,0.70,4.03,0.0838,0.1229
2,late_1_3d,1357,2.07,3.51,0.1887,0.2564
3,late_3_7d,1773,5.12,2.32,0.5302,0.6125
4,late_7_14d,1744,9.98,1.74,0.6806,0.7810
5,late_14d_plus,1504,28.04,1.71,0.6935,0.7866


**观察与总结**

本节将delay_days按延迟程度划分为多个区间，用于观察延迟时间增加时评分和差评率是否呈现阶梯式变化。

从分组结果看，准时或提前订单的评分表现最好；随着延迟天数增加，平均评分整体下降，1星差评率和低分率整体上升。尤其是延迟超过3天后，差评风险明显高于轻微延迟订单。

这一结果的业务意义在于，平台不应只在订单“已经严重延迟”后被动处理，而可以根据延迟天数设置预警阈值。例如，对延迟超过3天的订单提前触发通知、客服跟进或补偿策略，以降低差评风险。

## 6 地区履约与评分表现

In [8]:
state_delivery = (
    delivery_base
    .groupby("customer_state", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        avg_delivery_days=("delivery_days", "mean"),
        avg_delay_days=("delay_days", "mean"),
        late_rate=("is_late", "mean"),
        avg_review_score=("review_score", "mean"),
        one_star_rate=("is_one_star", "mean"),
    )
)

state_delivery = state_delivery[state_delivery["orders"] >= 500].copy()

state_delivery[
    ["avg_delivery_days", "avg_delay_days", "avg_review_score"]
] = state_delivery[
    ["avg_delivery_days", "avg_delay_days", "avg_review_score"]
].round(2)

state_delivery[["late_rate", "one_star_rate"]] = (
    state_delivery[["late_rate", "one_star_rate"]]
    .round(4)
)

state_delivery = state_delivery.sort_values(
    ["late_rate", "one_star_rate"],
    ascending=False,
)

display(state_delivery.head(10))

,customer_state,orders,avg_delivery_days,avg_delay_days,late_rate,avg_review_score,one_star_rate
9,MA,712,21.44,-8.97,0.1924,3.83,0.1573
5,CE,1273,21.21,-10.16,0.1524,3.94,0.1351
4,BA,3229,19.25,-10.20,0.1369,3.93,0.1316
18,RJ,12211,15.24,-11.11,0.1328,3.97,0.1471
13,PA,933,23.64,-13.51,0.1190,3.91,0.1436
7,ES,1969,15.61,-9.98,0.1188,4.08,0.1097
11,MS,699,15.63,-10.33,0.1159,4.16,0.1016
14,PB,512,20.25,-12.79,0.1074,4.08,0.1094
15,PE,1579,18.37,-12.67,0.1064,4.08,0.1159
23,SC,3519,14.88,-10.87,0.0963,4.13,0.0980


**观察与总结**

本节按customer_state统计各地区的订单量、平均配送时长、延迟率、平均评分和1星差评率，并筛选订单量达到一定规模的地区，以避免小样本造成比例波动。

从结果看，不同地区之间的履约表现存在差异，部分地区同时表现出较高的延迟率和较高的1星差评率。这说明履约问题可能具有区域特征，可能与配送距离、物流网络覆盖、当地履约能力等因素有关。

地区维度的分析可以帮助平台确定运营优先级。后续如果某些地区既是高销售区域，又存在较高延迟率和差评率，就应优先纳入物流优化和客户干预范围。

## 7 统计检验与阶段总结
统计检验只能说明延迟状态与评分表现之间存在显著相关关系，不能直接证明物流延迟是低评分的唯一原因。因为历史观察数据中还可能存在地区、品类、卖家服务质量等混杂因素。后续如果要验证主动干预策略的效果，需要通过A/B测试进一步确认。

### 7.1 Mann-Whitney U检验：评分整体是否不同
判断准时或提前订单的review_score分布，和延迟订单的review_score分布是否存在显著差异？Mann-Whitney U检验用于比较准时或提前订单与延迟订单的评分分布。由于review_score是1到5星的等级评分，不是连续正态变量，因此使用非参数检验更稳妥。

In [10]:
from scipy.stats import mannwhitneyu

on_time_scores = delivery_base.loc[
    ~delivery_base["is_late"],
    "review_score",
].dropna()

late_scores = delivery_base.loc[
    delivery_base["is_late"],
    "review_score",
].dropna()

u_stat, score_p_value = mannwhitneyu(
    on_time_scores,
    late_scores,
    alternative="two-sided",
)

print("U统计量:", u_stat)
print("评分分布差异p值:", f"{score_p_value:.2e}")

U统计量: 524484849.5
评分分布差异p值: 0.00e+00


**观察与总结**

Mann-Whitney U检验用于比较准时或提前订单与延迟订单的review_score分布是否存在差异。检验结果显示p值极小，可以拒绝“两组评分分布无显著差异”的原假设，说明准时或提前订单与延迟订单的评分分布存在显著差异。

需要注意的是，由于样本量较大，p值极小并不意味着该结果本身就具有足够业务价值，后续仍需结合平均评分差异、1星差评率差异和相对风险进行解释。

### 7.2 卡方检验：1星差评比例是否不同
订单是否延迟，和订单是否出现1星差评之间是否存在统计关联？

In [13]:
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(
    delivery_base["is_late"],
    delivery_base["is_one_star"],
)

display(contingency_table)

chi2_stat, one_star_p_value, dof, expected = chi2_contingency(
    contingency_table
)

expected_df = pd.DataFrame(
    expected,
    index=contingency_table.index,
    columns=contingency_table.columns
)

display(expected_df)

print("卡方统计量:", chi2_stat)
print("1星差评关联检验p值:", f"{one_star_p_value:.2e}")

is_one_star,False,True
is_late,,
False,82390,5779
True,4122,3533


is_one_star,False,True
is_late,,
False,79600.898814,8568.101186
True,6911.101186,743.898814


卡方统计量: 12583.900417831548
1星差评关联检验p值: 0.00e+00


**观察与总结**

卡方检验用于判断is_late和is_one_star这两个分类变量是否相互独立。原假设为“订单是否延迟与是否产生1星差评相互独立，不存在统计关联”。

检验结果显示p值极小，因此拒绝原假设，说明订单是否延迟与是否产生1星差评之间存在显著统计关联。结合交叉表可以看出，延迟订单中的1星差评比例明显高于准时或提前订单。

### 7.3 效应量分析：差异到底有多大

In [14]:
on_time_mask = ~delivery_base["is_late"]
late_mask = delivery_base["is_late"]

on_time_avg_score = delivery_base.loc[
    on_time_mask,
    "review_score",
].mean()

late_avg_score = delivery_base.loc[
    late_mask,
    "review_score",
].mean()

on_time_one_star_rate = delivery_base.loc[
    on_time_mask,
    "is_one_star",
].mean()

late_one_star_rate = delivery_base.loc[
    late_mask,
    "is_one_star",
].mean()

effect_summary = (
    pd.Series(
        {
            "on_time_avg_score": on_time_avg_score,
            "late_avg_score": late_avg_score,
            "avg_score_diff": late_avg_score - on_time_avg_score,
            "on_time_one_star_rate": on_time_one_star_rate,
            "late_one_star_rate": late_one_star_rate,
            "one_star_rate_diff": (
                late_one_star_rate - on_time_one_star_rate
            ),
            "one_star_relative_risk": (
                late_one_star_rate / on_time_one_star_rate
            ),
        }
    )
    .round(4)
    .to_frame("value")
)

display(effect_summary)

,value
on_time_avg_score,4.2943
late_avg_score,2.5651
avg_score_diff,-1.7292
on_time_one_star_rate,0.0655
late_one_star_rate,0.4615
one_star_rate_diff,0.3960
one_star_relative_risk,7.0414


**观察与总结**

从效应量看，延迟订单的平均评分约为2.57，准时或提前订单约为4.29，延迟组平均评分低约1.73分。相比单纯的显著性检验，这个差异更能体现履约延迟在业务上的影响程度。

从1星差评率看，准时或提前订单的1星差评率约为6.55%，延迟订单约为46.15%，相差约39.60个百分点。延迟订单出现1星差评的风险约为准时或提前订单的7.04倍，说明履约延迟与极端负面评价之间存在很强的相关关系。

相比于显著性检验，具体的评分下降幅度、差评率提升幅度和相对风险，这些指标更适合转化为业务预警和运营干预策略。

##总结

本阶段基于已送达且评分完整的95824个订单，分析了履约时效与客户评分之间的关系。

从整体履约表现看，样本平均配送时长约12.52天，中位数约10.21天；整体延迟率约7.99%。这说明大多数订单能够提前或按时送达，但仍有一定比例订单发生延迟。

从准时订单与延迟订单对比看，准时或提前订单的平均评分约4.29，延迟订单约2.57，延迟组低约1.73分。准时或提前订单的1星差评率约6.55%，延迟订单约46.15%，延迟订单出现1星差评的风险约为准时或提前订单的7.04倍。该结果说明延迟履约与客户极端负面评价之间存在明显相关关系。

从延迟天数分组看，轻微延迟对评分的影响相对有限，但延迟超过3天后，平均评分明显下降，1星差评率快速上升。该结果可以作为业务预警阈值的参考：平台可优先关注延迟超过3天的订单，并对延迟超过7天的订单采取更强的客服或补偿措施。

从地区维度看，不同customer_state之间的延迟率、配送天数和评分表现存在差异，说明履约风险具有一定区域特征。后续可以结合03中的地区销售贡献，优先识别高GMV、高延迟率、高差评率的重点区域。

统计检验结果进一步支持准时订单与延迟订单在评分分布和1星差评率上存在显著差异。但由于本项目使用的是历史观察数据，当前结论只能说明相关关系，不能直接证明物流延迟是低评分的唯一原因。若平台希望验证主动通知、优惠券补偿或客服干预是否能降低差评率，需要进一步设计A/B测试。